In [ ]:
import sys
import os

current_dir = os.getcwd()
src_dir = os.path.dirname(current_dir)
sys.path.append(src_dir)

import polars as pl
from pipeline.config import PipelineConfig, DataConfig, ProcessingConfig, StorageConfig, RayConfig, S3Location
from pipeline.pipeline import Pipeline
from data_preprocessing.data_normalization import BMLLNormalizer
from data_preprocessing.data_access import DataAccessFactory

In [ ]:
# Loop to run pipeline with different parameter combinations
import itertools

run_parameter_sweep = False  # Set to True to run full parameter sweep

if run_parameter_sweep:
    memory_multipliers = [2, 4, 6]
    memory_per_core_gbs = [2, 4]
    cpu_buffers = [0, 2, 4, 6, 8]
    flat_core_counts = [None, 4, 8]  # None for dynamic, or fixed core counts
    param_combinations = itertools.product(memory_multipliers, memory_per_core_gbs, cpu_buffers, flat_core_counts)
else:
    # Single configuration for testing
    param_combinations = [(2, 4, 1, None)]  # memory_multiplier, memory_per_core_gb, cpu_buffer, flat_core_count

res_array = []
for memory_multiplier, memory_per_core_gb, cpu_buffer, flat_core_count in param_combinations:
    import time
    start_time = time.time()
    try:
        print(f"Running with parameters: memory_multiplier={memory_multiplier}, memory_per_core_gb={memory_per_core_gb}, cpu_buffer={cpu_buffer}, flat_core_count={flat_core_count}")
        config = PipelineConfig(
            region='us-east-1',
            data=DataConfig(
                raw_data_path='s3://bmlldata',
                start_date='2024-01-02',
                end_date='2024-01-02',
                exchanges=['ARCX'],
                data_types=['trades']
            ),
            processing=ProcessingConfig(
                normalization=BMLLNormalizer()
            ),
            storage=StorageConfig(
                raw_data=S3Location(path='s3://bmlldata'),
                normalized=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
                features=S3Location(path='s3://orderflowanalysis/intermediate/features'),
                models=S3Location(path='s3://orderflowanalysis/output/models'),
                predictions=S3Location(path='s3://orderflowanalysis/output/predictions'),
                backtest=S3Location(path='s3://orderflowanalysis/output/backtest')
            ),
            ray=RayConfig(runtime_env={"working_dir": src_dir},
                          flat_core_count=flat_core_count,
                          memory_multiplier=memory_multiplier,
                          memory_per_core_gb=memory_per_core_gb,
                          max_retries=5,
                          cpu_buffer=cpu_buffer,
                          file_sort_order="desc")
        )
        
        pipeline = Pipeline(config)
        results = None
        results = pipeline.run(slice(1))
        end_time = time.time()
        run_time = end_time - start_time
        for r in results:
            r['memory_multiplier'] = memory_multiplier
            r['memory_per_core_gb'] = memory_per_core_gb
            r['cpu_buffer'] = cpu_buffer
            r['flat_core_count'] = flat_core_count
            r['total_run_time'] = run_time
        print(f"SUCCESS: Processed {len(results)} files with memory_multiplier={memory_multiplier}, memory_per_core_gb={memory_per_core_gb}, cpu_buffer={cpu_buffer}, flat_core_count={flat_core_count} in {run_time:.2f}s")
        res_array.append(results)
    except Exception as e:
        end_time = time.time()
        run_time = end_time - start_time
        import traceback
        error_result = {
            'memory_multiplier': memory_multiplier,
            'memory_per_core_gb': memory_per_core_gb,
            'cpu_buffer': cpu_buffer,
            'flat_core_count': flat_core_count,
            'total_run_time': run_time,
            'message': str(e),
            'error_type': type(e).__name__,
            'traceback': traceback.format_exc(),
            'stage': 'pipeline_execution'
        }
        res_array.append([error_result])
        print(f"EXCEPTION: Error with memory_multiplier={memory_multiplier}, memory_per_core_gb={memory_per_core_gb}, cpu_buffer={cpu_buffer}, flat_core_count={flat_core_count} in {run_time:.2f}s: {e}")

In [ ]:
# 3. Verify retry functionality
import pandas as pd
res_pd = pd.DataFrame(results)
successful = res_pd.query("message == 'success'").shape[0]
failed = res_pd.query("message != 'success'").shape[0]
print(f"Successful: {successful}, Failed: {failed}")

In [ ]:
# 4. Inspect paths
result = results[0]
print(f"Input:  {result['input_path']}")
print(f"Output: {result['output_path']}")
print(f"Rows:   {result['row_count']:,}")

In [ ]:
# 5. Read output
data_access = DataAccessFactory.create('s3', region='us-east-1')
output_data = data_access.read(result['output_path']).limit(100).collect()
print(f"✓ Read {len(output_data)} records")
output_data.head()

In [ ]:
# 6. Validate schema
normalizer = BMLLNormalizer()
expected_schema = normalizer.get_schema('trades')
missing = set(expected_schema.keys()) - set(output_data.columns)
assert not missing, f"Missing columns: {missing}"
print(f"✓ Schema validated ({len(output_data.columns)} columns)")
print("\n✓ All tests passed!")

In [ ]:
# 7. Compare raw input files with normalized output files
import pandas as pd

# Get list of raw input files
raw_files = pipeline._discover_files()
raw_paths = [f[0] for f in raw_files]

# Get list of normalized output files using discovery
norm_data_access = DataAccessFactory.create('s3', region='us-east-1')
norm_files = config.processing.normalization.discover_files(
    norm_data_access,
    config.storage.normalized.path,
    config.ray.file_sort_order
)
norm_paths = [f[0] for f in norm_files]

print(f"Raw files: {len(raw_paths)}")
print(f"Normalized files: {len(norm_paths)}")

# Extract file identifiers for comparison
def extract_file_id(path):
    # Extract YYYY/MM/DD/data_type/AMERICAS/filename pattern
    parts = path.split('/')
    if len(parts) >= 6:
        return '/'.join(parts[-6:])  # Last 6 parts: YYYY/MM/DD/type/region/file
    return path.split('/')[-1]  # Fallback to filename

raw_ids = {extract_file_id(p): p for p in raw_paths}
norm_ids = {extract_file_id(p): p for p in norm_paths}

# Find differences
raw_only = set(raw_ids.keys()) - set(norm_ids.keys())
norm_only = set(norm_ids.keys()) - set(raw_ids.keys())
common = set(raw_ids.keys()) & set(norm_ids.keys())

print(f"\nCommon files: {len(common)}")
print(f"Raw only: {len(raw_only)}")
print(f"Normalized only: {len(norm_only)}")

if raw_only or norm_only:
    diff_data = []
    for file_id in raw_only:
        diff_data.append({'file_id': file_id, 'status': 'raw_only', 'path': raw_ids[file_id]})
    for file_id in norm_only:
        diff_data.append({'file_id': file_id, 'status': 'norm_only', 'path': norm_ids[file_id]})
    
    diff_df = pd.DataFrame(diff_data)
    print("\nDifferences:")
    display(diff_df)
else:
    print("\n✓ All raw files have corresponding normalized files")